*Module 2 of 9*

> **¿Prefieres español?** Abre [`02_como_ve_un_satelite.ipynb`](../es/02_como_ve_un_satelite.ipynb) — es el mismo módulo, en español.


# 🛰️ Module 2 — How a satellite sees the world

🧭 **Objectives** — understand what a satellite actually measures, why an
image has many *bands*, what *reflectance* and a *spectral signature* are,
and the three *resolutions* that describe any sensor. Then open a **real**
tile of the Yaqui Valley and confirm it is exactly what Module 1 promised:
a grid of numbers.

📚 **The idea.** A satellite carries a **sensor** that measures how much
sunlight the ground **reflects** back, in several **bands** — narrow slices
of the electromagnetic spectrum. Our eyes see three bands (red, green,
blue). Satellites like **Landsat** and **Sentinel-2** see those *and* others
we cannot, especially **near-infrared (NIR)** and **short-wave infrared
(SWIR)**, where vegetation, soil and water differ the most.

The value stored for each pixel and band is **reflectance**: a fraction
between 0 and 1 of the light that bounced back (files store it as an integer
to save space, e.g. `4500` = 0.45). Plot reflectance across bands for one
pixel and you get its **spectral signature** — a fingerprint that says
"this is a thriving crop" or "this is bare soil".

![how a satellite sees](../../anim/en/01_where_it_runs.svg)


## Resolution: three ways to say "how detailed?"

Every sensor is described by three resolutions — remember them, they decide
what you can and cannot map:

- **Spatial**: how big is one pixel on the ground? Our tile is **30 m** per
  pixel (one pixel ≈ a small orchard). Finer = smaller fields visible.
- **Spectral**: how many bands, and how narrow? More bands = more chemistry
  you can read. Our tile has **6 spectral bands** plus derived indices.
- **Temporal**: how often does the satellite revisit? Every few days for
  Sentinel-2. This is what lets us watch a crop **grow** over a season.

📚 The data below is a **geomedian** (Module 3 explains it) of March 2018,
built from **NASA HLS** — Harmonized Landsat + Sentinel-2 — at 30 m.


In [ ]:
# Get the workshop tile (a few MB; cached after the first download)
import os, sys

async def get_file(name):
    for cand in (f"files/{name}", name, f"../files/{name}", f"../../files/{name}"):
        if os.path.exists(cand):
            return cand
    dest = f"/tmp/{name}"
    if not os.path.exists(dest):
        url = f"https://raw.githubusercontent.com/abxda/portable-geocrop/main/files/{name}"
        if sys.platform == "emscripten":
            from pyodide.http import pyfetch
            resp = await pyfetch(url)
            open(dest, "wb").write(await resp.bytes())
        else:
            import urllib.request
            urllib.request.urlretrieve(url, dest)
    return dest

TILE = await get_file("crop_tile_384.tif")
print("Tile ready:", TILE)

## Open the tile and read its shape

Just like the invented 8x8 array in Module 1 — but real. `rasterio` opens
GeoTIFF satellite files; `.read()` hands us a NumPy array with shape
`(bands, rows, cols)`.


In [ ]:
import numpy as np
import rasterio
import matplotlib.pyplot as plt

with rasterio.open(TILE) as src:
    img = src.read()                     # (13, 384, 384) integer array
    band_names = list(src.descriptions)  # what each layer is

print("Array shape (bands, rows, cols):", img.shape)
print("Data type:", img.dtype)
print("The 13 layers:", band_names)
print("One pixel (row 200, col 200), all bands:", img[:, 200, 200])

## See it in true color

The first three spectral bands are blue, green, red. Stack them (red, green,
blue order for display) and scale to 0–1, exactly the divide-by-a-number
trick from Module 1. This is the field as your eyes would see it from space.


In [ ]:
# Bands 0,1,2 = blue, green, red. Display wants Red-Green-Blue.
rgb = np.clip(np.dstack([img[2], img[1], img[0]]) / 3000.0, 0, 1)

plt.figure(figsize=(7, 7))
plt.imshow(rgb)
plt.title("Yaqui Valley — true color (geomedian, March 2018)")
plt.axis("off")
plt.show()
print("Every field you see is a patch of ~30 m pixels.")

## The spectral signature: light fingerprints

Now the payoff. Pick two pixels — one on a green field, one on bare soil —
and plot their reflectance across the 6 spectral bands. The curves are their
**spectral signatures**. Notice the crop's jump into the NIR band: that gap
is invisible to your eyes but obvious to the satellite, and it is the whole
basis of vegetation indices in Module 4.


In [ ]:
# The 6 spectral bands are layers 0..5 (blue,green,red,nir,swir1,swir2)
spectral = ["blue", "green", "red", "nir", "swir1", "swir2"]

# NDVI (layer 6) helps us find a green pixel and a bare one automatically
ndvi = img[6] / 10000.0
green_rc = np.unravel_index(np.argmax(ndvi), ndvi.shape)   # most vegetated
soil_rc  = np.unravel_index(np.argmin(np.where(ndvi > 0, ndvi, 9)), ndvi.shape)

crop_sig = img[0:6, green_rc[0], green_rc[1]] / 10000.0
soil_sig = img[0:6, soil_rc[0],  soil_rc[1]]  / 10000.0

plt.figure(figsize=(7, 4))
plt.plot(spectral, crop_sig, marker="o", color="green", label="green field")
plt.plot(spectral, soil_sig, marker="s", color="peru",  label="bare soil")
plt.ylabel("reflectance"); plt.title("Spectral signatures of two real pixels")
plt.legend(); plt.show()
print("See the crop leap up at NIR — that is chlorophyll, not color.")

## 🧪 Check yourself

**Your eyes see 3 bands. Why does a crop-mapping satellite bother measuring
near-infrared and SWIR, which we cannot see?**

<details><summary>Show answer</summary>

Because that is where surfaces differ most. Healthy vegetation reflects a
lot of NIR (from leaf structure) while absorbing red; soil and water behave
differently again. Those invisible bands carry the information that
separates crops from everything else — the visible colors alone are not
enough.

</details>

**A tile is 30 m spatial resolution. What does that number mean, and which
resolution lets us watch a crop grow through the season?**

<details><summary>Show answer</summary>

30 m spatial resolution means each pixel covers a 30 m × 30 m patch of
ground. Watching growth over time is **temporal** resolution — how often the
satellite revisits the same place (every few days for Sentinel-2).

</details>


## 🔭 Go deeper

Optional: these bilingual concept cards expand what you just learned
(prerequisite chains, lineage to fundamentals, curated references):

- [Remote sensing, the field itself](https://abxda.github.io/rs-learning-audio/?id=remote-sensing)
- [The electromagnetic spectrum](https://abxda.github.io/rs-learning-audio/?id=electromagnetic-spectrum)
- [Reflectance](https://abxda.github.io/rs-learning-audio/?id=reflectance)
- [The spectral signature](https://abxda.github.io/rs-learning-audio/?id=spectral-signature)
- [Spectral bands](https://abxda.github.io/rs-learning-audio/?id=spectral-bands)
- [Resolution (spatial/spectral/temporal)](https://abxda.github.io/rs-learning-audio/?id=resolution)
- [Landsat](https://abxda.github.io/rs-learning-audio/?id=landsat)
- [The Sentinel missions](https://abxda.github.io/rs-learning-audio/?id=sentinel-missions)



---

[← Previous · Module 1 — Just enough Python](01_just_enough_python.ipynb) · [Next → · Module 3 — Clean data: from clouds to the geomedian](03_clean_data_geomedian.ipynb)
